In [1]:
!pip install flask pyjwt cryptography flask-sqlalchemy

In [3]:
import os
from flask import Flask, request, jsonify, render_template_string
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash, check_password_hash
from cryptography.fernet import Fernet
import jwt
import datetime
from functools import wraps

app = Flask(__name__)

# --- CONFIGURATION ---
app.config['SECRET_KEY'] = 'super-secret-key-123'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///wearable_portal.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

# Encryption Key for Wearable Vitals
ENCRYPTION_KEY = Fernet.generate_key()
cipher = Fernet(ENCRYPTION_KEY)
db = SQLAlchemy(app)

# --- MODELS ---
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)
    password = db.Column(db.String(120), nullable=False)

class WearableData(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, db.ForeignKey('user.id'))
    encrypted_vitals = db.Column(db.Text, nullable=False)
    timestamp = db.Column(db.DateTime, default=datetime.datetime.utcnow)

# Initialize Database
with app.app_context():
    db.create_all()
    print("Database and Tables created.")

Database and Tables created.


In [4]:
def token_required(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        token = request.headers.get('Authorization')
        if not token:
            return jsonify({'message': 'Token is missing!'}), 401
        try:
            # Clean "Bearer " prefix if present
            actual_token = token.split(" ")[1] if " " in token else token
            data = jwt.decode(actual_token, app.config['SECRET_KEY'], algorithms=["HS256"])
            current_user = User.query.filter_by(id=data['user_id']).first()
        except Exception as e:
            return jsonify({'message': 'Token is invalid or expired!'}), 401
        return f(current_user, *args, **kwargs)
    return decorated

print("Auth middleware active.")

Auth middleware active.


In [5]:
@app.route('/api/signup', methods=['POST'])
def signup():
    data = request.get_json()
    hashed_pw = generate_password_hash(data['password'], method='pbkdf2:sha256')
    new_user = User(username=data['username'], password=hashed_pw)
    try:
        db.session.add(new_user)
        db.session.commit()
        return jsonify({'message': 'User created successfully!'})
    except:
        return jsonify({'message': 'Username already exists!'}), 400

@app.route('/api/login', methods=['POST'])
def login():
    auth = request.get_json()
    user = User.query.filter_by(username=auth['username']).first()
    if user and check_password_hash(user.password, auth['password']):
        token = jwt.encode({
            'user_id': user.id,
            'exp': datetime.datetime.utcnow() + datetime.timedelta(hours=24)
        }, app.config['SECRET_KEY'])
        return jsonify({'token': token})
    return jsonify({'message': 'Invalid credentials'}), 401

@app.route('/api/sync', methods=['POST'])
@token_required
def sync_data(current_user):
    data = request.get_json()
    # Data is encrypted before being stored in the database
    encrypted_data = cipher.encrypt(str(data).encode()).decode()
    new_record = WearableData(user_id=current_user.id, encrypted_vitals=encrypted_data)
    db.session.add(new_record)
    db.session.commit()
    return jsonify({'message': 'Vitals encrypted & saved.'})

@app.route('/api/data', methods=['GET'])
@token_required
def get_data(current_user):
    records = WearableData.query.filter_by(user_id=current_user.id).all()
    results = []
    for r in records:
        # Data is decrypted only when requested by the authenticated owner
        decrypted = cipher.decrypt(r.encrypted_vitals.encode()).decode()
        results.append({"time": r.timestamp.strftime("%H:%M:%S"), "vitals": eval(decrypted)})
    return jsonify(results)

In [ ]:
import os
from flask import Flask, request, jsonify, render_template_string
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash, check_password_hash
from cryptography.fernet import Fernet
import jwt, datetime, pandas as pd
from functools import wraps

# 1. KILL PREVIOUS INSTANCE & SETUP
app = Flask(__name__)
app.view_functions = {} # This fixes the AssertionError

app.config['SECRET_KEY'] = 'health-vault-2026'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///sync.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

ENCRYPTION_KEY = Fernet.generate_key()
cipher = Fernet(ENCRYPTION_KEY)
db = SQLAlchemy(app)

# 2. DATABASE MODELS
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True)
    password = db.Column(db.String(120))

class HealthData(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, db.ForeignKey('user.id'))
    payload = db.Column(db.Text) # Encrypted
    dt = db.Column(db.DateTime, default=datetime.datetime.utcnow)

with app.app_context():
    db.create_all()

# 3. AUTH MIDDLEWARE
def token_required(f):
    @wraps(f)
    def dec(*args, **kwargs):
        token = request.headers.get('Authorization')
        if not token: return jsonify({'m': 'No Token'}), 401
        try:
            data = jwt.decode(token, app.config['SECRET_KEY'], algorithms=["HS256"])
            curr = User.query.get(data['u'])
        except: return jsonify({'m': 'Bad Token'}), 401
        return f(curr, *args, **kwargs)
    return dec

# 4. API ENDPOINTS
@app.route('/api/auth', methods=['POST'])
def auth():
    d = request.get_json()
    u = User.query.filter_by(username=d['u']).first()
    if u and check_password_hash(u.password, d['p']):
        tk = jwt.encode({'u': u.id, 'exp': datetime.datetime.utcnow()+datetime.timedelta(hours=5)}, app.config['SECRET_KEY'])
        return jsonify({'tk': tk})
    # Auto-Register if user doesn't exist
    new = User(username=d['u'], password=generate_password_hash(d['p'], method='pbkdf2:sha256'))
    db.session.add(new)
    db.session.commit()
    return jsonify({'m': 'Account Created. Login again!'})

@app.route('/api/sync', methods=['POST'])
@token_required
def sync_data(curr):
    vitals = request.get_json()
    enc = cipher.encrypt(str(vitals).encode()).decode()
    db.session.add(HealthData(user_id=curr.id, payload=enc))
    db.session.commit()
    return jsonify({'m': 'Encrypted & Synced'})

@app.route('/api/fetch', methods=['GET'])
@token_required
def fetch_data(curr):
    rows = HealthData.query.filter_by(user_id=curr.id).all()
    res = []
    for r in rows:
        plain = eval(cipher.decrypt(r.payload.encode()).decode())
        res.append({'t': r.dt.strftime('%H:%M'), 'h': plain['h'], 's': plain['s']})
    return jsonify(res)

# 5. FRONTEND UI
@app.route('/')
def home():
    return render_template_string("""
    <!DOCTYPE html><html><head><title>Vault</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
    <style>body{background:#0f172a;color:#e2e8f0;font-family:sans-serif;}.card{background:#1e293b;border:none;border-radius:1rem;padding:2rem;margin-top:2rem;}</style></head>
    <body><div class="container" style="max-width:600px;"><div class="card">
    <div id="login-box"><h3>Portal Access</h3><input id="un" class="form-control mb-2" placeholder="User"><input type="password" id="pw" class="form-control mb-3" placeholder="Pass">
    <button class="btn btn-primary w-100" onclick="login()">Login / Register</button></div>
    <div id="dash" class="d-none"><h3>Live Wearable Sync</h3><div class="row text-center mb-4"><div class="col"><h6>HR</h6><h2 id="l-h" class="text-danger">--</h2></div>
    <div class="col"><h6>Steps</h6><h2 id="l-s" class="text-info">--</h2></div></div>
    <button class="btn btn-success w-100 mb-4" onclick="snc()">Sync Wearable</button><canvas id="graph"></canvas></div>
    </div></div>
    <script>
    let token = ''; let chart = null;
    async function login(){
        let r = await fetch('/api/auth',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({u:un.value,p:pw.value})});
        let d = await r.json();
        if(d.tk){token=d.tk;document.getElementById('login-box').classList.add('d-none');document.getElementById('dash').classList.remove('d-none');ref();}
        else{alert(d.m);}
    }
    async function snc(){
        await fetch('/api/sync',{method:'POST',headers:{'Content-Type':'application/json','Authorization':token},
        body:JSON.stringify({h:Math.floor(70+Math.random()*30),s:Math.floor(Math.random()*1000)})});
        ref();
    }
    async function ref(){
        let r = await fetch('/api/fetch',{headers:{'Authorization':token}});
        let d = await r.json();
        if(d.length > 0){
            let last = d[d.length-1]; document.getElementById('l-h').innerText = last.h; document.getElementById('l-s').innerText = last.s;
            if(chart)chart.destroy();
            chart = new Chart(document.getElementById('graph'),{type:'line',data:{labels:d.map(x=>x.t),
            datasets:[{label:'HR',data:d.map(x=>x.h),borderColor:'#f43f5e',tension:0.4}]}});
        }
    }
    </script></body></html>
    """)

# 6. RUN
from google.colab.output import eval_js
print(f"Your Secure Portal is live at: {eval_js('google.colab.kernel.proxyPort(5000)')}")
app.run(port=5000)

Your Secure Portal is live at: https://5000-m-s-11w6aq04fx2b1-a.us-east4-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [12/Mar/2026 16:32:19] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [12/Mar/2026 16:32:19] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [12/Mar/2026 16:32:32] "POST /api/auth HTTP/1.1" 200 -
